# Streaming plans: satisfaction mosaic plot

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Mosaic Plot** · a worked example from the StatNex teaching guide

**The story.** A streaming service surveyed 619 subscribers on its Basic (with ads), Standard and Premium plans about their satisfaction, on a five-point scale.

**The question.** How does satisfaction differ between the three plans?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `11_chart/mosaic_plot.csv` (3 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Plan,Very dissatisfied,Dissatisfied,Neutral,Satisfied,Very satisfied
Basic,38.0,56.0,70.0,51.0,20.0
Standard,9.0,20.0,58.0,92.0,64.0
Premium,8.0,15.0,36.0,48.0,34.0

These cells save `totals.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile totals.py
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.contingency_tables import Table

pd.set_option("display.width", 120)

## The data

In [ ]:
# A table of counts: one row per stratum, one column per category. A blank or text
# cell is a count of 0 -- a category nobody fell into is still a category.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
categories = ["Very dissatisfied", "Dissatisfied", "Neutral", "Satisfied", "Very satisfied"]
labels = data["Plan"].astype(str)
# A row or column named total or sum, or a Japanese word for a total (listed in totals.py), is a margin, not a category:
# it is dropped, or every count would be counted twice. Spaces, underscores, case and
# full-width letters are ignored when names are compared.
from totals import is_total

categories = [name for name in categories if not is_total(name)]
data, labels = data[~labels.map(is_total)], labels[~labels.map(is_total)]
counts = data[categories].apply(pd.to_numeric, errors="coerce").fillna(0)
counts.index = labels
total = counts.to_numpy().sum()
print(counts)

## Composition within each stratum

In [ ]:
# Each row divided by its total: the heights of the rectangles. A row with no counts
# gets zeros.
composition = counts.div(counts.sum(axis=1), axis=0).fillna(0)
print()
print(composition)

## Column widths

In [ ]:
# Each stratum's share of the grand total: the widths of the bars, so that a
# rectangle's area is its count.
widths = pd.DataFrame({"total": counts.sum(axis=1), "width": counts.sum(axis=1) / total})
print()
print(widths)

## Adjusted residuals

In [ ]:
# How far each cell is from what independence predicts, standardised: beyond +-1.96
# at 5%, +-2.58 at 1%. These are the cells whose segments fail to line up across bars.
# They are Haberman's adjusted residuals, (observed - expected) / its standard error,
# statsmodels' Table.standardized_resids (shift_zeros=False keeps the counts as they
# are). Many cells are read at once, so a lone residual just past 2 is weak evidence;
# the chi-square test below asks about the table as a whole.
with np.errstate(divide="ignore", invalid="ignore"):
    adjusted = pd.DataFrame(Table(counts.to_numpy(float), shift_zeros=False).standardized_resids,
                            index=counts.index, columns=counts.columns)
print()
print(adjusted)

## Test of independence

In [ ]:
# Pearson's chi-square on the table without any empty row or column: is the
# composition the same in every stratum? It needs most expected counts of 5 or more.
# A small P says the strata differ somewhere, not how much; the residuals show where.
nonempty = counts.loc[counts.sum(axis=1) > 0, counts.sum(axis=0) > 0]
chi2, p, df, _ = stats.chi2_contingency(nonempty, correction=False)
print(f"\nchi-square = {chi2:.4f}, df = {df}, P = {p:.4g}")

## Mosaic plot

In [ ]:
# Bars as wide as each stratum's share, split by the composition within it.
Path("figures").mkdir(exist_ok=True)
colours = plt.cm.Blues(np.linspace(0.25, 0.9, len(categories)))
fig, ax = plt.subplots(figsize=(8, 6))
left = 0.0
for i, width in enumerate(widths["width"]):
    bottom = 0.0
    for j, category in enumerate(categories):
        height = composition.iat[i, j]
        alpha = 0.82
        ax.bar(left, height, width=width, bottom=bottom, align="edge", color=colours[j],
               alpha=alpha, edgecolor="white", label=category if i == 0 else None)
        text = f"{height:.1%}"
        if height > 0:
            ax.text(left + width / 2, bottom + height / 2, text, ha="center", va="center",
                    fontsize=7)
        bottom += height
    left += width
ax.set_xticks(np.cumsum(widths["width"]) - widths["width"] / 2, counts.index)
ax.legend(fontsize=8, bbox_to_anchor=(1, 1), loc="upper left")
fig.savefig("figures/mosaic.png", dpi=120, bbox_inches="tight")

## What the result says

In a mosaic plot each plan is one column. The column's width shows how many subscribers the plan has, and the tiles inside show the share giving each answer. On Basic, 16.2% are very dissatisfied and 23.8% are dissatisfied. On Standard the same shares are 3.7% and 8.2%, and on Premium 5.7% and 10.6%.

The adjusted residual says how far a cell is from what we would expect if plan and satisfaction were unrelated. Values far from 0 mark real differences. For Basic the two dissatisfied cells have residuals of about 5.0, and "Very satisfied" has -5.2. For Premium the largest residual is only 1.74.

Basic subscribers are clearly the least satisfied. Standard and Premium look similar, and this survey cannot tell them apart.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.